# Correlation Coefficient — Obesity vs Diabetes
### All 58 California Counties · CDC PLACES Data (2022–2023)

**What are we studying?**
- 58 data points — one for each California county
- Each dot on the scatter plot represents ONE county
- X-axis: the percentage of adults in that county who are obese (the predictor)
- Y-axis: the percentage of adults in that county who have diabetes (the outcome)

**Why obesity on X and diabetes on Y?**

Research generally shows that obesity increases the risk of developing type 2 diabetes — not the other way around. So we place obesity as the independent variable (X) and diabetes as the dependent variable (Y). We're asking: "As obesity goes up in a county, what tends to happen to diabetes?"

**Where does this data come from?**

The CDC (Centers for Disease Control and Prevention) runs a program called **PLACES** that estimates health measures for every county in the United States. They don't survey every county directly — instead, they take state-level survey data from the **Behavioral Risk Factor Surveillance System (BRFSS)** and use statistical models to estimate county-level rates.

**Citation:**
> Greenlund KJ, Lu H, Wang Y, et al. *PLACES: Local Data for Better Health.* Prev Chronic Dis 2022;19:210459.  
> Data: https://data.cdc.gov

---

## Step 1: Load the Data

We load the full CDC PLACES dataset and filter it to California counties, looking at just two health measures: **Obesity** and **Diabetes**.

The dataset contains TWO types of prevalence for each county:

| Type | What it means | Analogy |
|------|--------------|--------|
| **Crude (Raw %)** | The actual percentage — no adjustments. If 300 out of 1,000 adults are obese, the crude rate is 30%. | Raw test scores |
| **Age-Adjusted** | What the rate *would be* if every county had the same age mix. Removes the effect of one county having more seniors than another. | Grading on a curve to make it fair |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Load the full CDC dataset
df = pd.read_csv("cdc_places_data.csv", low_memory=False)

# Filter: California only, Diabetes and Obesity only
df_ca = df[
    (df["StateAbbr"] == "CA") &
    (df["MeasureId"].isin(["DIABETES", "OBESITY"]))
].copy()

print(f"California rows for Obesity & Diabetes: {len(df_ca):,}")
print(f"Counties: {df_ca['LocationName'].nunique()}")
print(f"Data Value Types: {df_ca['DataValueTypeID'].unique()}")

## Step 2: Separate Crude and Age-Adjusted Data

We split the data into two tables — one for crude prevalence, one for age-adjusted. Each table has 58 rows (one per county) and two columns (Obesity %, Diabetes %).

In [ ]:
def build_pivot(df_ca, data_type):
    """Pivot one data value type into county x measure format."""
    subset = df_ca[df_ca["DataValueTypeID"] == data_type]
    pivot = subset.pivot_table(
        index="LocationName",
        columns="MeasureId",
        values="Data_Value",
        aggfunc="mean",
    ).dropna()
    return pivot

crude_df = build_pivot(df_ca, "CrdPrv")
age_adj_df = build_pivot(df_ca, "AgeAdjPrv")

print(f"Crude prevalence:        {len(crude_df)} counties")
print(f"Age-adjusted prevalence: {len(age_adj_df)} counties")
print()
print("Sample of CRUDE data (first 5 counties):")
print(crude_df.head())
print()
print("Sample of AGE-ADJUSTED data (first 5 counties):")
print(age_adj_df.head())

## Step 3: Calculate the Correlation Coefficient

The **Pearson correlation coefficient (r)** measures the strength and direction of a linear relationship between two variables.

- **r = 1.0** → perfect positive correlation (as one goes up, the other always goes up)
- **r = 0.0** → no correlation (no relationship)
- **r = -1.0** → perfect negative correlation (as one goes up, the other always goes down)

| r value | Strength |
|---------|----------|
| 0.8 – 1.0 | Very Strong |
| 0.6 – 0.8 | Strong |
| 0.4 – 0.6 | Moderate |
| 0.2 – 0.4 | Weak |
| 0.0 – 0.2 | Very Weak / None |

In [ ]:
# Calculate Pearson r for both datasets
r_crude, p_crude = stats.pearsonr(crude_df["OBESITY"], crude_df["DIABETES"])
r_adj, p_adj = stats.pearsonr(age_adj_df["OBESITY"], age_adj_df["DIABETES"])

print("CORRELATION COEFFICIENTS")
print("=" * 40)
print(f"  Crude (Raw %):   r = {r_crude:.4f}")
print(f"  Age-Adjusted:    r = {r_adj:.4f}")
print("=" * 40)

## Step 4: Interactive Scatter Plot — Toggle Between Views

Use the dropdown below to switch between **Crude** and **Age-Adjusted** data.

Watch what happens:
- The dots shift position (some counties move more than others)
- The correlation coefficient (r) changes
- The description below the chart explains what you're seeing

In [ ]:
from IPython.display import display, HTML
try:
    import ipywidgets as widgets
except ImportError:
    !pip install ipywidgets -q
    import ipywidgets as widgets


def plot_scatter(view_type):
    """Draw scatter plot for the selected data type."""

    if view_type == "Crude (Raw %)":
        data = crude_df
        r_val = r_crude
        dot_color = "#E07A3A"
        description = (
            "CRUDE (RAW %) — What you see is what you get\n\n"
            "This is the actual percentage of adults with each condition. "
            "If 300 out of 1,000 adults in Fresno County are obese, "
            "the crude rate is 30%. No adjustments — just the raw number.\n\n"
            "Keep in mind: Some counties have older populations, which "
            "naturally increases obesity and diabetes rates. So part of what "
            "you're seeing might just be age differences between counties, "
            "not true health differences."
        )
    else:
        data = age_adj_df
        r_val = r_adj
        dot_color = "#2E6B8A"
        description = (
            "AGE-ADJUSTED — Apples to apples comparison\n\n"
            "This answers: 'What would each county's rate be if they all "
            "had the same age mix?' The CDC statistically removes the "
            "effect of having an older or younger population.\n\n"
            "Now we're comparing fairly. If a county STILL shows high "
            "obesity after age-adjustment, it's not just because of older "
            "residents — something else is going on (diet, income, access "
            "to healthcare, physical activity, etc.)."
        )

    # Classify strength
    a = abs(r_val)
    if a >= 0.8:
        strength = "Very Strong"
    elif a >= 0.6:
        strength = "Strong"
    elif a >= 0.4:
        strength = "Moderate"
    elif a >= 0.2:
        strength = "Weak"
    else:
        strength = "Very Weak / None"

    # ── Create the figure ──
    fig, ax = plt.subplots(figsize=(12, 8))

    # Scatter dots — OBESITY on X, DIABETES on Y
    ax.scatter(
        data["OBESITY"], data["DIABETES"],
        s=80, alpha=0.7, color=dot_color,
        edgecolors="white", linewidths=0.8, zorder=3,
    )

    # Label every county
    for county, row in data.iterrows():
        ax.annotate(
            county, (row["OBESITY"], row["DIABETES"]),
            fontsize=7, alpha=0.7, color="#555",
            xytext=(5, 4), textcoords="offset points",
        )

    # Titles and labels
    ax.set_title(
        f"Obesity vs Diabetes — 58 California Counties\n"
        f"Correlation Coefficient:  r = {r_val:.4f}  ({strength} Positive Correlation)",
        fontsize=14, fontweight="bold", pad=15,
    )
    ax.set_xlabel("Obesity Prevalence — % of Adults", fontsize=12, labelpad=10)
    ax.set_ylabel("Diabetes Prevalence — % of Adults", fontsize=12, labelpad=10)

    # Grid
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.set_xlim(14, 40)
    ax.set_ylim(6.5, 16.5)

    # Stats box
    stats_text = (
        f"n = {len(data)} counties\n"
        f"r = {r_val:.4f}"
    )
    ax.text(
        0.98, 0.04, stats_text,
        transform=ax.transAxes, fontsize=11,
        fontfamily="monospace", verticalalignment="bottom",
        horizontalalignment="right",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="white",
                  edgecolor="#ccc", alpha=0.9),
    )

    # Source
    fig.text(
        0.5, -0.02,
        "Data: CDC PLACES — Local Data for Better Health (2022–2023)  ·  "
        "Greenlund KJ et al., Prev Chronic Dis 2022;19:210459  ·  data.cdc.gov",
        ha="center", fontsize=8, color="#aaa", style="italic",
    )

    plt.tight_layout()
    plt.show()

    # Print description below the chart
    print("─" * 60)
    print(description)
    print("─" * 60)


# ── Create the toggle dropdown ──
toggle = widgets.Dropdown(
    options=["Crude (Raw %)", "Age-Adjusted"],
    value="Crude (Raw %)",
    description="View:",
    style={"description_width": "50px"},
    layout=widgets.Layout(width="250px"),
)

output = widgets.interactive_output(plot_scatter, {"view_type": toggle})

display(toggle, output)

## Step 5: Compare the Two Views Side by Side

Here we see both scatter plots at the same time so we can directly compare how the dots shift when we remove the age effect.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

for ax, data, r_val, title, color in [
    (ax1, crude_df, r_crude, "Crude (Raw %)", "#E07A3A"),
    (ax2, age_adj_df, r_adj, "Age-Adjusted", "#2E6B8A"),
]:
    # OBESITY on X, DIABETES on Y
    ax.scatter(
        data["OBESITY"], data["DIABETES"],
        s=70, alpha=0.7, color=color,
        edgecolors="white", linewidths=0.8, zorder=3,
    )

    for county, row in data.iterrows():
        ax.annotate(
            county, (row["OBESITY"], row["DIABETES"]),
            fontsize=6, alpha=0.6, color="#555",
            xytext=(4, 3), textcoords="offset points",
        )

    ax.set_title(f"{title}\nr = {r_val:.4f}", fontsize=14, fontweight="bold")
    ax.set_xlabel("Obesity Prevalence — % of Adults", fontsize=10)
    ax.set_ylabel("Diabetes Prevalence — % of Adults", fontsize=10)
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.set_xlim(14, 40)
    ax.set_ylim(6.5, 16.5)

fig.suptitle(
    "Obesity vs Diabetes — 58 California Counties\n"
    "How does the correlation coefficient change when we adjust for age?",
    fontsize=15, fontweight="bold", y=1.02,
)

fig.text(
    0.5, -0.02,
    "Data: CDC PLACES (2022–2023)  ·  Greenlund KJ et al., Prev Chronic Dis 2022;19:210459",
    ha="center", fontsize=8, color="#aaa", style="italic",
)

plt.tight_layout()
plt.show()

print()
print("WHAT CHANGED?")
print("=" * 50)
print(f"  Crude r:        {r_crude:.4f}")
print(f"  Age-Adjusted r: {r_adj:.4f}")
print(f"  Difference:     {r_adj - r_crude:+.4f}")
print()
print("The age-adjusted r is slightly HIGHER. Why?")
print("Because age was adding 'noise' to the relationship.")
print("Some counties looked like they had high obesity")
print("and high diabetes just because they had more seniors.")
print("Once we remove that age effect, the underlying")
print("connection between obesity and diabetes becomes")
print("a little clearer.")

## Discussion Questions

1. **What does r = 0.63 (or 0.67) tell us?**  
   It tells us there's a strong positive association — counties with higher obesity rates tend to also have higher diabetes rates.

2. **Does this mean obesity CAUSES diabetes?**  
   Not from this data alone! Correlation tells us two things move together — it does NOT tell us that one causes the other. We placed obesity on the X-axis because research suggests it's a risk factor for diabetes, but this scatter plot alone cannot prove causation.

3. **What else might cause BOTH obesity and diabetes to be high in the same counties?**  
   Think about: poverty, food deserts (limited access to healthy food), less access to healthcare, fewer parks and places to exercise, cultural factors, education levels.

4. **Why did r go UP when we age-adjusted?**  
   Because age was a *confounding variable*. Older people are more likely to have both conditions. Some counties appeared to have high rates simply because they had older populations. Removing that effect revealed a slightly stronger underlying relationship.

5. **Can you find your county on the chart? Where does it fall?**

---

*Data: CDC PLACES — Local Data for Better Health, County Data (2022–2023 Release)*  
*Citation: Greenlund KJ, Lu H, Wang Y, et al. PLACES: Local Data for Better Health. Prev Chronic Dis 2022;19:210459.*

## Step 6: Explore — What Else Is Connected to Obesity?

We just saw that obesity and diabetes are strongly correlated (r = 0.67). But we also asked: **does obesity cause diabetes, or is something else driving both?**

The CDC dataset has 40 health measures for every California county. Let's explore some of them. Use the dropdown below to pick a measure and see how it correlates with obesity.

As you click through, ask yourself:
- Which measures have the **strongest** correlation with obesity?
- Do you notice a **pattern** in what's strongly correlated?
- What does that pattern tell us about **why** some counties have more obesity than others?
- Are there any **surprises** — things you expected to correlate that don't?

In [ ]:
# ── Build the full age-adjusted pivot table with all measures ──

adj_all = df[df["DataValueTypeID"] == "AgeAdjPrv"]
adj_ca = adj_all[adj_all["StateAbbr"] == "CA"]

pivot_all = adj_ca.pivot_table(
    index="LocationName",
    columns="MeasureId",
    values="Data_Value",
    aggfunc="mean",
).dropna(axis=1, how="all")

print(f"Full pivot table: {pivot_all.shape[0]} counties × {pivot_all.shape[1]} measures")
print(f"Measures available: {sorted(pivot_all.columns.tolist())}")

In [ ]:
# ── Curated measures with hand-written descriptions ──

EXPLORE_MEASURES = {
    "DIABETES": {
        "label": "Diabetes",
        "y_label": "Diabetes Prevalence — % of Adults",
        "description": (
            "You already studied this one. Counties with more obesity tend to have more 
"
            "diabetes. But is obesity really the cause, or is something else driving both?"
        ),
    },
    "LPA": {
        "label": "Physical Inactivity",
        "y_label": "Physical Inactivity — % of Adults",
        "description": (
            "Less exercise, more obesity — this one feels obvious. But why are some 
"
            "counties less active? Is it because people choose not to exercise, or because 
"
            "they don't have safe parks, sidewalks, or free time?"
        ),
    },
    "FOODSTAMP": {
        "label": "Food Stamps",
        "y_label": "Food Stamps — % of Adults Receiving Assistance",
        "description": (
            "This is the percentage of adults receiving food assistance. Why would this be 
"
            "so strongly connected to obesity? Think about what kinds of food are cheapest 
"
            "and most available in low-income areas."
        ),
    },
    "FOODINSECU": {
        "label": "Food Insecurity",
        "y_label": "Food Insecurity — % of Adults",
        "description": (
            "People who aren't sure where their next meal is coming from. Notice how 
"
            "strongly this tracks with obesity — the 'obesity is a choice' argument 
"
            "starts to break down here."
        ),
    },
    "HOUSINSECU": {
        "label": "Housing Insecurity",
        "y_label": "Housing Insecurity — % of Adults",
        "description": (
            "People worried about paying rent or mortgage. What does housing have to do 
"
            "with obesity? Maybe when you're stressed about keeping a roof over your head, 
"
            "healthy eating isn't the top priority."
        ),
    },
    "LACKTRPT": {
        "label": "Transportation Barriers",
        "y_label": "Transportation Barriers — % of Adults",
        "description": (
            "People who can't get where they need to go. No transportation might mean no 
"
            "access to grocery stores with fresh food, no way to get to a gym, and no way 
"
            "to get to a doctor."
        ),
    },
    "BPHIGH": {
        "label": "High Blood Pressure",
        "y_label": "High Blood Pressure — % of Adults",
        "description": (
            "Another health consequence that clusters with obesity. The same counties keep 
"
            "showing up at the high end of these charts."
        ),
    },
    "CHD": {
        "label": "Coronary Heart Disease",
        "y_label": "Coronary Heart Disease — % of Adults",
        "description": (
            "Heart disease follows the same pattern. These health outcomes are piling up in 
"
            "the same places — what do those places have in common?"
        ),
    },
    "BINGE": {
        "label": "Binge Drinking",
        "y_label": "Binge Drinking — % of Adults",
        "description": (
            "Surprise — almost no correlation. Not everything is connected to obesity. 
"
            "This is what r near 0 looks like. The dots are scattered with no clear 
"
            "pattern."
        ),
    },
    "DENTAL": {
        "label": "Dental Visits",
        "y_label": "Dental Visits — % of Adults",
        "description": (
            "This one goes the opposite direction — a negative correlation. Counties with 
"
            "more obesity have fewer dental visits. Dental care is expensive and often not 
"
            "covered by insurance. Another poverty signal."
        ),
    },
}

print(f"Curated measures for exploration: {len(EXPLORE_MEASURES)}")
for key, info in EXPLORE_MEASURES.items():
    print(f"  {info["label"]:<28} ({key})")

In [ ]:
# ── Interactive Explorer: Pick a measure, see the scatter plot ──

def r_strength(r):
    a = abs(r)
    if a >= 0.8: return "Very Strong"
    elif a >= 0.6: return "Strong"
    elif a >= 0.4: return "Moderate"
    elif a >= 0.2: return "Weak"
    else: return "Very Weak / None"


def explore_measure(measure_name):
    """Draw scatter plot of Obesity vs selected measure."""

    # Find the measure key from the label
    measure_key = None
    for key, info in EXPLORE_MEASURES.items():
        if info["label"] == measure_name:
            measure_key = key
            break

    info = EXPLORE_MEASURES[measure_key]
    clean = pivot_all[["OBESITY", measure_key]].dropna()
    r_val, p_val = stats.pearsonr(clean["OBESITY"], clean[measure_key])

    # Determine direction
    if r_val > 0.2:
        direction = "Positive"
        dot_color = "#E07A3A"
    elif r_val < -0.2:
        direction = "Negative"
        dot_color = "#2E6B8A"
    else:
        direction = "No"
        dot_color = "#888888"

    strength = r_strength(r_val)

    # ── Create the figure ──
    fig, ax = plt.subplots(figsize=(12, 8))

    # Scatter — OBESITY always on X
    ax.scatter(
        clean["OBESITY"], clean[measure_key],
        s=80, alpha=0.7, color=dot_color,
        edgecolors="white", linewidths=0.8, zorder=3,
    )

    # Label every county
    for county in clean.index:
        ax.annotate(
            county,
            (clean.loc[county, "OBESITY"], clean.loc[county, measure_key]),
            fontsize=7, alpha=0.7, color="#555",
            xytext=(5, 4), textcoords="offset points",
        )

    # Title with r value
    ax.set_title(
        f"Obesity vs {info["label"]} — 58 California Counties
"
        f"Correlation Coefficient:  r = {r_val:.4f}  ({strength} {direction} Correlation)",
        fontsize=14, fontweight="bold", pad=15,
    )
    ax.set_xlabel("Obesity Prevalence — % of Adults", fontsize=12, labelpad=10)
    ax.set_ylabel(info["y_label"], fontsize=12, labelpad=10)

    # Grid
    ax.grid(True, linestyle="--", alpha=0.3)

    # Stats box
    stats_text = f"n = {len(clean)} counties
r = {r_val:.4f}"
    ax.text(
        0.98, 0.04, stats_text,
        transform=ax.transAxes, fontsize=11,
        fontfamily="monospace", verticalalignment="bottom",
        horizontalalignment="right",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="white",
                  edgecolor="#ccc", alpha=0.9),
    )

    # Source
    fig.text(
        0.5, -0.02,
        "Data: CDC PLACES — Local Data for Better Health (2022–2023)  ·  "
        "Greenlund KJ et al., Prev Chronic Dis 2022;19:210459  ·  data.cdc.gov",
        ha="center", fontsize=8, color="#aaa", style="italic",
    )

    plt.tight_layout()
    plt.show()

    # Print the hand-written description
    print("─" * 60)
    print(f"{info["label"].upper()} vs OBESITY")
    print(f"r = {r_val:.4f}  |  {strength} {direction} Correlation")
    print("─" * 60)
    print()
    print(info["description"])
    print()
    print("─" * 60)


# ── Build the dropdown ──
measure_labels = [info["label"] for info in EXPLORE_MEASURES.values()]

explore_toggle = widgets.Dropdown(
    options=measure_labels,
    value=measure_labels[0],
    description="Measure:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="300px"),
)

explore_output = widgets.interactive_output(explore_measure, {"measure_name": explore_toggle})

display(explore_toggle, explore_output)

## Step 7: What Did We Discover?

After exploring the measures above, a pattern emerges:

**The strongest correlations with obesity aren't just other diseases — they're signs of poverty.**

| Measure | r | What it tells us |
|---------|---|------------------|
| High Blood Pressure | +0.89 | Health consequence — clusters with obesity |
| Coronary Heart Disease | +0.86 | Health consequence — same counties, same problems |
| Transportation Barriers | +0.83 | Poverty signal — can't get to stores, gyms, doctors |
| Food Stamps | +0.83 | Poverty signal — low income |
| Housing Insecurity | +0.79 | Poverty signal — can't afford stable housing |
| Physical Inactivity | +0.77 | Behavior — but driven by environment? |
| Food Insecurity | +0.77 | Poverty signal — not enough food access |
| Dental Visits | -0.77 | Poverty signal (negative) — can't afford dental care |
| Diabetes | +0.67 | Health consequence — where we started |
| Binge Drinking | -0.14 | No correlation — not everything is connected |

### The Big Takeaway

When we started, we saw that obesity and diabetes were correlated and asked: **does obesity cause diabetes?**

Now we can see that the same counties with high obesity and high diabetes also have high food insecurity, high housing insecurity, more people on food stamps, and fewer dental visits. These are all markers of **poverty**.

This suggests that **poverty may be the confounding variable** — it doesn't just cause obesity OR diabetes. It creates conditions (cheap processed food, no safe places to exercise, limited healthcare access, chronic stress) that lead to BOTH.

This is why **correlation does not equal causation**. The scatter plot showed us that obesity and diabetes move together, but it took deeper exploration to see that a third factor — economic hardship — may be driving both.

---

*Data: CDC PLACES — Local Data for Better Health, County Data (2022–2023 Release)*  
*Citation: Greenlund KJ, Lu H, Wang Y, et al. PLACES: Local Data for Better Health. Prev Chronic Dis 2022;19:210459.*